# Experiment 1.3.10 — stacked-bin SNN ablation

This notebook is **analysis-only**: it does not train SNNs. Training is performed by `scripts/experiment_1_3_10_stacked_bin_snn_ablation.py` through the Slurm array launcher.

Fixed input representation: `[256,30] -> [16 bins,16 within-bin steps,30] -> [16,480]`. All conditions use whole-output-spike-count inference.

Factorial design: 3 architectures (`1h128`, `1h256`, `2h128`) × 2 objectives (`whole_count_ce`, `timestep_ce`) × 2 dynamics regimes (`weight_only`, `trainable_dynamics`) × 3 user-disjoint split seeds (`11`, `23`, `101`) = **36 runs**.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'snn').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

REPO_ROOT = find_repo_root()
EXPERIMENT_ID = 'experiment_1_3_10_stacked_bin_snn_ablation'
PROTOCOL_VERSION = 'stacked250_v1'
LABELS = tuple(sorted(('A','B','C','D','E','X','G','H','I','J','K','L')))
LABEL_TAG = 'labels_' + '-'.join(LABELS)
RESULTS_DIR = REPO_ROOT / 'notebooks' / 'artifacts' / EXPERIMENT_ID / PROTOCOL_VERSION / LABEL_TAG
RUNS_DIR = RESULTS_DIR / 'runs'
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_SEEDS = (11,23,101)
ARCHITECTURES = ('1h128','1h256','2h128')
OBJECTIVES = ('whole_count_ce','timestep_ce')
TRAIN_REGIMES = ('weight_only','trainable_dynamics')
EXPECTED_RUNS = 36
print('Results:', RESULTS_DIR)

## Load run artifacts and verify pairing

In [ ]:
def flatten_run(p):
    row = {
        'run_key': p['run_key'], 'split_seed': int(p['split_seed']),
        'architecture': p['architecture'], 'objective': p['objective'],
        'train_regime': p['train_regime'], 'best_epoch': int(p['best_epoch']),
        'model_seed': int(p['model_seed']), 'sample_hash': p['sample_hash'],
        'trainable_parameters': int(p['parameter_counts']['trainable']),
    }
    for split in ('train','val','test'):
        m = p['metrics'][split]
        for key in ('loss','balanced_accuracy','accuracy','macro_f1','output_firing_rate'):
            row[f'{split}_{key}'] = float(m[key])
        for i, fr in enumerate(m.get('hidden_firing_rates', []), 1):
            row[f'{split}_hidden{i}_firing_rate'] = float(fr)
    for hidden in p['best_dynamics']['hidden']:
        layer = hidden['layer']
        for q in ('tau_syn_ms','tau_mem_ms','threshold'):
            row[f'{layer}_{q}_mean'] = float(hidden[q]['mean'])
            row[f'{layer}_{q}_std'] = float(hidden[q]['std'])
    return row

run_files = sorted(RUNS_DIR.glob('run_*.json')) if RUNS_DIR.exists() else []
payloads = [json.loads(p.read_text()) for p in run_files]
runs = pd.DataFrame(flatten_run(p) for p in payloads)
if not runs.empty:
    runs = runs.sort_values(['split_seed','architecture','train_regime','objective']).reset_index(drop=True)
print(f'Completed: {len(runs)}/{EXPECTED_RUNS}')
display(runs.head())
if not runs.empty:
    assert (runs.groupby('split_seed').sample_hash.nunique() == 1).all(), 'Split pairing failed'
    assert (runs.groupby(['split_seed','architecture']).model_seed.nunique() == 1).all(), 'Initialization pairing failed'

## Aggregate performance across user splits

In [ ]:
if runs.empty:
    summary = pd.DataFrame()
else:
    summary = (runs.groupby(['architecture','train_regime','objective'], as_index=False)
        .agg(n_splits=('split_seed','nunique'),
             mean_val_ba=('val_balanced_accuracy','mean'), sd_val_ba=('val_balanced_accuracy','std'),
             mean_test_ba=('test_balanced_accuracy','mean'), sd_test_ba=('test_balanced_accuracy','std'),
             mean_test_accuracy=('test_accuracy','mean'), sd_test_accuracy=('test_accuracy','std'),
             mean_test_macro_f1=('test_macro_f1','mean'), sd_test_macro_f1=('test_macro_f1','std'),
             mean_best_epoch=('best_epoch','mean')))
display(summary)
if not summary.empty:
    summary.to_csv(RESULTS_DIR / 'notebook_summary.csv', index=False)

## Main comparison: architecture × objective, separated by dynamics regime

In [ ]:
if not summary.empty:
    x = np.arange(len(ARCHITECTURES))
    fig, axes = plt.subplots(1,2,figsize=(13,4.8),sharey=True)
    for ax, regime in zip(axes, TRAIN_REGIMES):
        for objective in OBJECTIVES:
            part = summary[(summary.train_regime==regime)&(summary.objective==objective)].set_index('architecture').reindex(ARCHITECTURES)
            ax.errorbar(x, part.mean_test_ba, yerr=part.sd_test_ba.fillna(0), marker='o', capsize=4, label=objective)
        ax.set_xticks(x, ARCHITECTURES); ax.set_xlabel('SNN architecture'); ax.set_title(regime); ax.grid(axis='y',alpha=.25)
    axes[0].set_ylabel('Test balanced accuracy'); axes[1].legend()
    fig.suptitle('Stacked-bin SNN architecture × objective'); fig.tight_layout()
    fig.savefig(FIGURES_DIR/'test_ba_architecture_objective_by_dynamics.png', dpi=180, bbox_inches='tight')
    plt.show()

## Paired effects of trainable dynamics and objective

In [ ]:
if not runs.empty:
    dyn = runs.pivot_table(index=['split_seed','architecture','objective'], columns='train_regime', values='test_balanced_accuracy', aggfunc='first').reset_index()
    if {'weight_only','trainable_dynamics'}.issubset(dyn.columns):
        dyn = dyn.dropna(subset=['weight_only','trainable_dynamics']).copy()
        dyn['delta_ba'] = dyn.trainable_dynamics - dyn.weight_only
        dyn_summary = dyn.groupby(['architecture','objective'],as_index=False).agg(mean_delta=('delta_ba','mean'),sd_delta=('delta_ba','std'))
        display(dyn_summary)
        dyn.to_csv(RESULTS_DIR/'paired_dynamics_deltas.csv',index=False)
        fig, ax = plt.subplots(figsize=(8,4.8)); x=np.arange(len(ARCHITECTURES))
        for j,obj in enumerate(OBJECTIVES):
            p=dyn_summary[dyn_summary.objective==obj].set_index('architecture').reindex(ARCHITECTURES)
            ax.errorbar(x+(j-.5)*.16,p.mean_delta,yerr=p.sd_delta.fillna(0),marker='o',linestyle='none',capsize=4,label=obj)
        ax.axhline(0,linewidth=1); ax.set_xticks(x,ARCHITECTURES); ax.set_ylabel('Δ test BA (trainable − fixed)'); ax.legend(); ax.grid(axis='y',alpha=.25)
        fig.tight_layout(); fig.savefig(FIGURES_DIR/'paired_trainable_dynamics_gain.png',dpi=180,bbox_inches='tight'); plt.show()

    obj = runs.pivot_table(index=['split_seed','architecture','train_regime'], columns='objective', values='test_balanced_accuracy', aggfunc='first').reset_index()
    if {'whole_count_ce','timestep_ce'}.issubset(obj.columns):
        obj = obj.dropna(subset=['whole_count_ce','timestep_ce']).copy()
        obj['delta_ba'] = obj.whole_count_ce - obj.timestep_ce
        obj_summary = obj.groupby(['architecture','train_regime'],as_index=False).agg(mean_delta=('delta_ba','mean'),sd_delta=('delta_ba','std'))
        display(obj_summary)
        obj.to_csv(RESULTS_DIR/'paired_objective_deltas.csv',index=False)

## Learned hidden-neuron dynamics at the best validation checkpoint

In [ ]:
rows=[]
for p in payloads:
    if p['train_regime']!='trainable_dynamics': continue
    for h in p['best_dynamics']['hidden']:
        rows.append({'split_seed':p['split_seed'],'architecture':p['architecture'],'objective':p['objective'],'layer':h['layer'],
                     'tau_syn_ms':h['tau_syn_ms']['mean'],'tau_mem_ms':h['tau_mem_ms']['mean'],'threshold':h['threshold']['mean']})
learned=pd.DataFrame(rows)
if not learned.empty:
    learned_summary=(learned.groupby(['architecture','objective','layer'],as_index=False)
        .agg(mean_tau_syn_ms=('tau_syn_ms','mean'),sd_tau_syn_ms=('tau_syn_ms','std'),
             mean_tau_mem_ms=('tau_mem_ms','mean'),sd_tau_mem_ms=('tau_mem_ms','std'),
             mean_threshold=('threshold','mean'),sd_threshold=('threshold','std')))
    display(learned_summary)
    learned_summary.to_csv(RESULTS_DIR/'learned_dynamics_summary.csv',index=False)
if not runs.empty:
    runs.to_csv(RESULTS_DIR/'notebook_all_runs.csv',index=False)